## ***Compiling and Optimizing a PyTorch Model***

***PyTorch feature can automatically convert model’s code to `TorchScript`, which is a `statically typed` subset of Python***

***Two main benefits:*** 
1. ***`TorchScript` code can be `compiled` and optimized to produce significantly faster models. For example, multiple operations can often be fused into a single, more efficient operation. Operations on constants (e.g., 2 * 3) can be replaced with their result (e.g., 6); this is called `constant folding`. Unused code can be pruned, and so on***
2. ***TorchScript can be `serialized`, saved to `disk`, and then loaded and executed in Python or in a `C++ environment` using the `LibTorch` library. This makes it possible to run PyTorch models on a wide range of devices, including embedded devices***

***2 ways to convert PyTorch model to TorchScript***
1. ***`Tracing`: PyTorch just runs your model with some `sample data`, `logs every operation that takes place`, and then converts this log to `TorchScript`. This is done using the `torch.jit.trace()` function***

    ***This generally works well with static models whose `forward()` method doesn’t use conditionals or loops***

2. ***`scripting`: PyTorch parses your Python code directly and converts it to `TorchScript`*** 
    
    ***This method supports `if statements` and `while` loops properly, as long as the conditions are tensors.It also supports `for` loops when iterating over tensors. it only works on a subset of Python, for example can't use `global variable`, `python generator`, `complex list comprehensions`, `variable length function arguments` (*args or **kwargs), or `match statements`***

    ***Moreover, types must be fixed (a function cannot return an integer in some cases and a float in others), and can only call other functions if they also respect these rules***

In [ ]:
torchscript_model = torch.jit.trace(model, x_new)

In [ ]:
torchscript_model = torch.jit.script(model)

***Regardless of whether you use tracing or scripting to produce your TorchScript model, you can then further optimize it***

In [ ]:
optimizer_model = torch.jit.optimizer_for_inference(torchscript_model)

***TorchScript models can only be used for inference, not for training, since the TorchScript environment doesn’t support gradient tracking or parameter updates***

***save a TorchScript model using its `save()` method***

In [ ]:
torchscript_model.save('my_fashion_mnist_torchscript.pt') 

***load it using the `torch.jit.load()` function***

In [ ]:
loaded_torchscript_model =  torch.jit.load("my_fashion_mnist_torchscript.pt") 

***TorchScript is no longer under active development—bugs are fixed but no new features are added***

> ***TorchScript is no longer under active development—bugs are fixed but no new features are added. It still works fine and it remains one of the best ways to run your PyTorch models in a C++ environment, but since the release of PyTorch 2.0 in March 2023, the PyTorch team has been focusing its efforts on a new set of compilation tools centered around the `torch.compile()` function***

In [ ]:
compiled_model = torch.compile(model) 

***The resulting model can now be used normally, and it will automatically be compiled and optimized when you use it***

***This is called Just-In-Time (JIT) compilation, as opposed to Ahead-Of-Time (AOT) compilation***

***Under the hood, `torch.compile()` relies on `TorchDynamo` (or Dynamo for short) which hooks directly into Python bytecode to capture the model’s computation graph at inference time***

***Having access to the `bytecode` allows `Dynamo` to efficiently and reliably capture the `computation graph`, properly handling conditionals and loops, while also benefiting from dynamic information that can be used to better optimize the model***

***The actual compilation and optimization is performed by default by a backend component named `TorchInductor`, which in turn relies on the `Triton language` to generate highly efficient `GPU code` (Nvidia only), or on the `OpenMP API` for `CPU optimization`***

***PyTorch `2.x` offers a few other optimization backends, including the `XLA backend` for Google’s TPU devices: just set `device="xla"` when calling `torch.compile()`***